登录

In [ ]:
import requests
import json
from os.path import expanduser
from requests.auth import HTTPBasicAuth

# load credentials
with open(expanduser('brain_credential_copy.txt')) as f:
    credentials = json.load(f)

# Extract username and password
username, password = credentials

# Create a session object
sess = requests.Session()

# Set up basic authentication
sess.auth = HTTPBasicAuth(username, password)

# Send a Post request to the API for authentication
response = sess.post('https://api.worldquantbrain.com/authentication')

# Print the response status and content for debugging
print(response.status_code)
print(response.json())

回测Alpha策略

In [ ]:
simulation_data={
    'type':'REGULAR',
    'settings':{
        'instrumentType':'EQUITY',
        'region':'USA',
        'universe':'TOP3000',
        'delay':1,
        'decay':0,
        'neutralization':'INDUSTRY',
        'truncation':0.04,
        'pasteurization':'ON',
        'unitHandling':'VERIFY',
        'nanHandling':'ON',
        'language':'FASTEXPR',
        'visualization':False,
    },
    'regular':'liabilities/assets'  #表达式

    }
    

In [ ]:
from time import sleep
sim_resp = sess.post(
    'https://api.worldquantbrain.com/simulations',
    json=simulation_data,
)
sim_progress_url =sim_resp.headers['Location']

while True:
    sim_progress_resp = sess.get(sim_progress_url)
    retry_after_sec=float(sim_progress_resp.headers.get('Retry-After', 0))
    if retry_after_sec == 0:
        break
    sleep(retry_after_sec)

alpha_id = sim_progress_resp.json()['alpha']

In [ ]:
alpha_id

In [ ]:
from time import time
def submit_alpha(s,alpha_id):
    brain_api_url='https://api.worldquantbrain.com'
    result=s.post(brain_api_url+'/alphas/'+alpha_id+'/submit')
    while True:
        if "retry-after" in result.headers:
            sleep(float(result.headers["Retry-After"]))
            result=s.get(brain_api_url+'/alphas/'+alpha_id)
        else:
            break
    return result.status_code==200
submittable_alpha_id='xAMx8NLg'
print(submit_alpha(sess,submittable_alpha_id))


In [ ]:
sim_progress_url

In [ ]:
sim_progress_resp.json()